Dorothee MG

Study: Validation of empathy videos\
-1- Targets are filmed talking about emotional life events\
-2- Targets watch their own videos and rate their emotional valence through time + identify specific emotions they felt\
-3- Perceivers watch targets' videos and rate them (same scales)
    
=> now we compare the targets and the perceiver evaluation
    
Goal: Calculate correspondance between behavioral data from participants and target (continuous evaluation + specific emotion identification)

In [5]:
# libraries 

import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from scipy.interpolate import interp1d

In [6]:
# lists

targets = ["P03_joy", "P29_joy", "P34_joy", "P42_joy", "P44_joy",
           "P12_sadness", "P15_sadness", "P27_sadness", "P34_sadness", "P44_sadness",
           "P03_anger", "P15_anger", "P22_anger", "P38_anger", "P42_anger", 
           "P17_pride", "P22_pride", "P29_pride", "P42_pride", "P44_pride",
           "P03_shame", "P22_shame", "P37_shame", "P38_shame", "P45_shame",
           "P12_neutral", "P17_neutral", "P29_neutral", "P34_neutral", "P38_neutral"]

perceivers = ["S01", "S02", "S03", "S04", "S05", "S06", "S07", "S08", "S11", "S15", "S18", "S20", "S23",
                "S24", "S28", "S34", "S37", "S42", "S44", "S45", "S47", "S49", "S51", "S54", "S55", "S57", "S64", "S65"]

# emotions on the Geneva Emotion Wheel (for specific emotion selection)
geneva_emotions = ['geneva_interest', 'geneva_amusement',	'geneva_pride',	'geneva_joy', 'geneva_pleasure',
                   'geneva_contentment',	'geneva_love',	'geneva_admiration',	'geneva_relief',	'geneva_compassion',
                   'geneva_sadness',	'geneva_guilt',	'geneva_regret',	'geneva_shame',	'geneva_disappointment',
                   'geneva_fear',	'geneva_disgust',	'geneva_contempt',	'geneva_hate',	'geneva_anger']

# smoothed continuous data (from previous script)
perceiver_data_continuous = pd.read_csv("outputs/gaussian_smoothed_perceiver.csv") # continuous evaluation data
target_data_continuous = pd.read_csv("outputs/gaussian_smoothed_target.csv")
perceiver_data_geneva = pd.read_csv("behavioral_data/PERCEIVER_video_ratings_geneva.csv") # Geneva Emotion Wheel data
target_data_geneva = pd.read_csv("behavioral_data/TARGET_video_ratings_geneva.csv")

empathic accuracy (EA) score  = correspondance between target and perceiver evaluation. There are 3 main 'methods'

-1- 'correlation': calculate correlation between target and perceiver evaluation through time (score between 1-9)\
-2- 'difference': calculate the absolute distance between target and perceiver evaluation through time (total average, score between 1-9)\
-3- 'geneva': calculate the recognition of specific emotions on the Geneva Emotion Wheel + distance between perceiver and target's evaluation of emotional intensity (intensity between 0-5 for each emotion on the wheel)

Here I use the 'smoothed' continuous data (gaussian sliding window of approximately 10s) created in script #1.\
I am also adding, in addition to EA scores, the 'shifted' EA scores (looking for best correlation/distance in the continuous evaluation; up to +/- 5s shift).

In [7]:

def run_analysis(method):
    
    all_output = [] # put in here scores from the perceiver-target pairs
    time_shifts = np.arange(-5, 5, 0.1)  # adding shifts in data of max. 10s (increments of 0.1s)
    
    for perceiver in perceivers:
        for target in targets:
            
            if method in ['correlation', 'difference']: # both use the same data
              
                perceiver_column = f"{perceiver}_{target}" # perceiver id + which target is being evaluated
                if perceiver_column not in perceiver_data_continuous.columns:
                    continue  # perceivers did not watch all targets
                
                target_emotion = target.split('_')[1] # will use this later
                perceiver_rating = perceiver_data_continuous[perceiver_column].dropna()

                for any_target in targets: # do the EA scores for all targets (matching and nonmatching ones)
                    any_target_emotion = any_target.split('_')[1] # will use this later
                    any_target_rating = target_data_continuous[any_target].dropna()
                    
                    # cutting to min duration between both videos (might use another method later?)
                    common_data = range(min(len(perceiver_rating), len(any_target_rating))) # creates a range from 0 to min duration between both videos
                    perceiver_rating_trimmed = perceiver_rating.loc[common_data] # selects rows of data up to the shortest length
                    any_target_rating_trimmed = any_target_rating.loc[common_data] # same thing for the target

                    # useful to interpolate shifted data (e.g., +100ms)
                    interpol = interp1d(np.arange(len(perceiver_rating_trimmed)), perceiver_rating_trimmed, kind='linear', fill_value='extrapolate')    
                                    # unsure here about fill_value: "points outside the data range will be extrapolated"
                    
                    output = {
                        "perceiver": perceiver, # id of the person who watched the target
                        "evaluation_of": target, # target that was evaluated
                        "compare_with_target": any_target, # target (specific or random) used for correlation calculation
                        "emotion": target_emotion, # the emotion of the target (the right one)
                        "match_type": "match" if target == any_target else "nonmatch", # match = specific target, else = nonmatch (surrogate)
                        "emotion_type": "same_emotion" if target_emotion == any_target_emotion else "diff_emotion"  # identify if the specific target and the surrogate target have the same emotion or not
                    }
                    
                    if method == 'correlation':
                        if (max(perceiver_rating_trimmed) - min(perceiver_rating_trimmed) < 0.5):
                            continue  # ignore cases where the perceiver didn't move the cursor (no correlation)

                        r_value, p_value = spearmanr(perceiver_rating_trimmed, any_target_rating_trimmed)
                        max_corr = r_value # if no better one, keep the original
                        best_time_shift = 0 # if no better one, keep the original
                        
                        for shift in time_shifts:
                            shifted_rating = interpol(np.arange(len(perceiver_rating_trimmed)) + shift) # interpolate perceiver rating
                            shifted_corr, shifted_p = spearmanr(shifted_rating, any_target_rating_trimmed)
                            if shifted_corr > max_corr: # if the correlation is better with the shift, save data
                                max_corr, max_p, best_time_shift = shifted_corr, shifted_p, shift
                                
                        output.update({
                            "spearman_r": r_value, # original r
                            "p_value": p_value,    # original p value
                            "max_spearman_r": max_corr, # max. correlation in 10s window
                            "max_p_value": max_p, # p value associated with the max. correlation
                            "time_shift_corr": best_time_shift}) # the time shift producing the best r 
                        
                    if method == 'difference':
                        initial_avg_diff = np.mean(np.abs(perceiver_rating_trimmed - any_target_rating_trimmed))
                        min_avg_diff = initial_avg_diff # if no better one, keep the original
                        
                        best_time_shift = 0 # if no better one, keep the original
                        for shift in time_shifts:
                            shifted_rating = interpol(np.arange(len(perceiver_rating_trimmed)) + shift)
                            avg_diff = np.mean(np.abs(shifted_rating - any_target_rating_trimmed)) # average over whole duration
                            if avg_diff < min_avg_diff:
                                min_avg_diff, best_time_shift = avg_diff, shift
                        
                        output.update({
                            "avg_difference": initial_avg_diff, 
                            "min_avg_difference": min_avg_diff, 
                            "time_shift_diff": best_time_shift})
                        
                    all_output.append(output)

            if method == 'geneva':
                target_emotion = target.split('_')[1]
                perceiver_row = perceiver_data_geneva[ # select the row of the evaluation
                    (perceiver_data_geneva['perceiver_id'] == perceiver) & 
                    (perceiver_data_geneva['target_id'] == target)]
                target_row = target_data_geneva[target_data_geneva['target_id'] == target]
                if perceiver_row.empty or target_row.empty: 
                    continue # perceivers did not watch all videos
                
                for any_target in targets:
                    any_target_row = target_data_geneva[target_data_geneva['target_id'] == any_target]
                    any_target_emotion = any_target.split('_')[1]
                    if any_target_row.empty:
                        continue # just in case
                    
                    sum_distance = 0 # total score /20 ; 1 point per recognized emotion (binary score)
                    sum_recognition = 0 # total score 100 - sum(distance between perceiver and target score for each given emotion)
                    for emotion in geneva_emotions:
                        perceiver_score = perceiver_row[emotion].values[0] # values for each evaluated emotion
                        any_target_score = any_target_row[emotion].values[0]

                        # distance and recognition
                        distance_score = abs(any_target_score - perceiver_score) # distance for each emotion between target and perceiver
                        recognition_score = 1 if ((any_target_score != 0 and perceiver_score != 0) or # recognition = 1 ; otherwise 0
                                                (any_target_score == 0 and perceiver_score == 0)) else 0

                        sum_distance += distance_score # add all scores together
                        sum_recognition += recognition_score

                    final_distance = 100 - sum_distance # so that better score = better performance
                    total_score = final_distance * sum_recognition # combination of both recognition and distance scores

                    output = {
                        "perceiver": perceiver,
                        "evaluation_of": target,
                        "compare_with_target": any_target,
                        "sum_distance": final_distance,
                        "sum_recognition": sum_recognition,
                        "final_score": total_score,
                        "match_type": "match" if target == any_target else "nonmatch",
                        "emotion": target_emotion,
                        "emotion_type": "same_emotion" if target_emotion == any_target_emotion else "diff_emotion"
                    }

                    all_output.append(output)
                
    pd.DataFrame(all_output).to_excel(f"outputs/{method}_EA_scores.xlsx", index=False)

In [8]:
for method in ['geneva', 'correlation', 'difference']:
    run_analysis(method)